# Railway timetabling

## Introduction
Efficient railway timetabling lies at the heart of reliable, high-capacity public transport systems, especially in densely used networks where small disruptions can propagate quickly. The **Periodic Event Scheduling Problem (PESP)** provides a powerful modeling framework for capturing the cyclic structure of timetables. Solving this type of problem involves interesting combinations of mixed-integer programming and graph algorithms.

This modeling example introduces a practical optimization-based approach to modeling railway timetables using PESP. Step-by-step we will implement two approaches for solving any PESP instance.

## Problem structure
In a PESP, each event is represented by a node in a directed graph. Each node should be assigned a time beween 0 and the cycle length $T$. That event is assumed to repeat every cycle at the same time (modulo $T$). Constraints are represented by arcs between two nodes, indicating the minimum and maximum difference between the times assigned to the connected nodes. 

If we would remove the cyclic aspect of this problem, solving this would be relatively simple task for which approaches like topological sorting could be used. The cyclic aspects make the problem both interesting and harder to solve. For example, if we have $T=60$ and two nodes connected by an arc with $[l_a,u_a]=[12, 17]$ then assigning times $t_1=53, t_2=5$ is valid too.

While this may seem like a rather abstract problem structure, it can be used for many real-life aspects of railway timetabling. Events usually denote the arrival or departure of a specific train service at a station or other important node in the physical network. Arcs indicate driving times (from departure to arrival), time spent at a station (from arrival to departure), turnaround time (from departure in one direction, to arrival in the opposite direction) or connection/transfer time (from arrival of one train, to departure of the other).

### Sample model
Throughout this modeling example, we will be working with a very small instance having 7 events, 10 arcs and a cycle time of $T=30$. The data can be found in `data/small.txt`. This problem is small enough to inspect by hand, but rich enough to explain every part of the algorithms. The nodes in the image show the node index (small text) and an example of an assigned time (large text). The arcs show the minimum and maximum difference between tail and head of the arc, as well as a weight which will be defined in the next section. For example, the constraint between nodes 4 and 0 are satisfied because $t_0=6$ occurs 12 minutes after $t_4=24$.

![Small problem instance](images/small.png)

## Formal definition
Let $V$ be a finite set of **events** and $A$ a set of directed **arcs**, forming a constraint graph $G = (V, A)$. Each arc $a = (i, j)$ carries a lower bound $l_a$, an upper bound $u_a$, and a non-negative weight $w_a$.

A **timetable** assigns each event a time $x_i \in [0, T)$. For arc $a = (i, j)$, the **periodic tension** is the unique value

$$z_a = x_j - x_i + T \cdot p_a \;\in\; [l_a,\; u_a]$$

where $p_a \in \mathbb{Z}$ is the integer that places $z_a$ in the correct range. The objective minimises the total weighted excess above the lower bounds:

$$\min \sum_{a \in A} w_a (z_a - l_a)$$

## Setup
Throughout this example, we will be using various common Python packages to represent problems, solutions and algorithms. We use type annotations to clarify the purpose of various variables more clearly.

In [87]:
EventId   = int   # event identifier (arbitrary int, as in the instance file)
ArcIndex  = int   # index into the arc arrays
Direction = int   # arc direction within a cycle: +1 (forward) or -1 (reversed)

We first describe what a PESP **instance** looks like. We're using numpy arrays here to allow for efficient matrix computations in the algorithms that solve our instances.

In [88]:
import numpy as np
from dataclasses import dataclass
from functools import cached_property

@dataclass
class PESPInstance:
    from_event:    np.ndarray
    to_event:      np.ndarray
    lower_bound:   np.ndarray
    upper_bound:   np.ndarray
    weight:        np.ndarray
    T: int = 60
    
    @property
    def n_arcs(self) -> int:
        return len(self.from_event)

    @cached_property
    def events(self) -> np.ndarray:
        """Sorted array of unique event IDs."""
        return np.union1d(self.from_event, self.to_event)

    @property
    def n_events(self) -> int:
        return len(self.events)
    
    def __repr__(self) -> str:
        return f"PESPInstance(n_arcs={self.n_arcs}, n_events={self.n_events}, T={self.T})"

A **solution** to PESP is an assignment of times to nodes. We simply store these times in a numpy array, assuming the same ordering as the arrays in the corresponding `PESPInstance`. We define a function `evaluate()` which checks feasibility and computes the objective value for the solution.

In [89]:
@dataclass
class Timetable:
    times: np.ndarray
    instance: PESPInstance
    
    def evaluate(self) -> tuple[float, bool]:
        instance = self.instance
        x_from = self.times[instance.from_event].astype(float)
        x_to   = self.times[instance.to_event].astype(float)
        lb, ub, w = instance.lower_bound, instance.upper_bound, instance.weight
        z = lb + (x_to - x_from - lb) % instance.T
        return float(w @ (z - lb)), bool((z <= ub).all())


## Instance files

We use the file format defined by [PESPlib](https://timpasslib.aalto.fi/pesplib.html), a public benchmark library of real-world periodic timetabling instances. The small sample files bundled with this notebook (`data/small.txt`, etc.) follow that same format. We present a simple utility function for parsing such a file and returning a `PESPInstance` representation. If you have a Gurobi license that allows solving models if arbitrary size, you can download official PESPlib instances and pass them directly to `parse_instance` without any changes. To keep our code simple, we convert all node IDs into zero-based numbers; if the data starts with node 1, we simply subtract 1 from each node ID we encounter.

The format is semicolon-separated text with one arc per line; lines starting with `#` are comments. Solution files record the objective in a `#obj:` header followed by one tension per arc.

*Instance format*

```text
# constraint_id; from_event; to_event; lower_bound; upper_bound; weight
1; 1; 2; 8; 13; 4
2; 2; 3; 9; 12; 1
```

*Solution format*

```text
#obj: 62
1; 13
2; 10
```

In [90]:
from pathlib import Path
import re
import pandas as pd

def parse_instance(path: str | Path, T: int = 60) -> PESPInstance:
    fe, te, lb, ub, w = [], [], [], [], []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [int(x.strip()) for x in line.split(";")]
        fe.append(parts[1]); te.append(parts[2])
        lb.append(parts[3]); ub.append(parts[4]); w.append(parts[5])
    offset = min(*fe, *te)
    return PESPInstance(
        from_event=np.array(fe,  dtype=int) - offset,
        to_event=np.array(te,    dtype=int) - offset,
        lower_bound=np.array(lb, dtype=float),
        upper_bound=np.array(ub, dtype=float),
        weight=np.array(w,       dtype=float),
        T=T,
    )


def parse_solution(path: str | Path) -> tuple[int, np.ndarray]:
    path = Path(path)
    with path.open() as fh:
        obj_line = next(l for l in fh if l.startswith("#obj:"))
    objective = int(re.search(r"-?\d+", obj_line).group())

    df = pd.read_csv(
        path,
        sep=r"\s*;\s*",
        comment="#",
        header=None,
        names=["constraint_id", "tension"],
        dtype=int
    )

    tension = df["tension"].to_numpy(dtype=float)
    return objective, tension

We can simply test the code by reading the small instance, reconstructing the solution from the image above and evaluating its objective function value and feasibility. We will see how our algorithms will find a solution wiht better objective value soon!

In [91]:
instance = parse_instance("data/small.txt", T=30)
print(instance)        
print(f"Weight range: [{instance.weight.min():.0f}, {instance.weight.max():.0f}]")

timetable = Timetable(np.array([0, 10, 20, 0, 12, 24, 4]), instance)
obj, feas = timetable.evaluate()
print(f"objective={obj:.0f}, feasible={feas}")

PESPInstance(n_arcs=10, n_events=7, T=30)
Weight range: [1, 4]
objective=69, feasible=True


## The arc-based approach

### Mathematical formulation
Earlier we introduced the arc constraint $z_a = x_j - x_i + T p_a \in [l_a, u_a]$. This shows the two types of variables involved: event times $x_i \in [0, T)$ for all nodes $i$ and an integer offset $p_a \in \mathbb{Z}$ per arc $a$. These are the only variables needed for this model formulation.

Throughout this example we will be using the Gurobi matrix-oriented API. The reason is that it's rather intuitive to preprocess our data into matrix format. We can use the **arc-node incidence matrix** $B$. Each row corresponds to an arc $a = (i,j)$ and each column corresponds to a node. For arc $a = (i,j)$, we set $B_{a,i} = -1$ and $B_{a,j} = +1$. In other words, the only columns having non-zero values in that row are the tail (-1) and head (+1) of the arc. 

If we multiply $B$ with a given solution $x$, we get a vector with length $|A|$ and individual elements $(Bx)_a = x_j - x_i$ for arc $a = (i,j)$. That means we can write the tension vector as $z = Bx + Tp$. 

Now we can formulate the problem as follows:

$$
\min_{x,\,p}\; \mathbf{w}^\top(z - \mathbf{l}) \\
\text{s.t.}\\
\quad \mathbf{l} \;\le\; z \;\le\; \mathbf{u} \\
z \;=\; Bx + Tp \; \\
x \in [0,T)^{|V|} \\ 
p \in \mathbb{Z}^{|A|}
$$

So far we defined $p$ allowing any integer value. But since $x_i, x_j \in [0, T)$, we know that the raw difference $x_j - x_i$ lies in $(-(T-1), T-1)$. Substituting this into $l_a \le x_j - x_i + T p_a \le u_a$ and isolating $p_a$ gives:

$$p_a \;\in\; \left[\left\lceil\frac{l_a - (T-1)}{T}\right\rceil,\; \left\lfloor\frac{u_a + (T-1)}{T}\right\rfloor\right]$$

For our sample with $T = 30$ these bounds reduce to $\{0, 1\}$ for nearly every arc.

### Implementation
The two implementations we will see in this modeling example, share the same lifecycle: build a Gurobi model, optionally solve and/or export it, then dispose of it. We capture this in an abstract base class.

The caller creates and owns a Gurobi environment `gp.Env`; the formulation creates a model of type `gp.Model` from that environment and owns only the model. Using the formulation as a context manager (`with`) guarantees the model is disposed even if an exception is raised.

In [92]:
from abc import ABC, abstractmethod
import gurobipy as gp
from gurobipy import GRB
try:
    from typing import Self
except ImportError:
    from typing_extensions import Self

class PESPFormulation(ABC):
    def __init__(self, env: gp.Env) -> None:
        self._model = gp.Model(env=env)        

    def __enter__(self) -> Self:
        return self

    def __exit__(self, *args: object) -> None:
        self._model.dispose()

    @abstractmethod
    def _extract(self) -> Timetable:
        pass

    def solve(self) -> Timetable:
        self._model.optimize()
        if self._model.Status != GRB.OPTIMAL:
            raise RuntimeError(f"Gurobi status {self._model.Status}")
        return self._extract()

    def export(self, path: str | Path) -> None:        
        self._model.write(str(path))

We will now show the implementation of the arc-based formulation as a subclass of `PESFormulation`. The constructor will compute $B$ and the bounds for $p$. Using that, it will construct the full Gurobi model. We also implement `_extract` to read the solution from `x.X` after `solve()` has been called.

In [93]:
from scipy.sparse import csr_matrix

class ArcFormulation(PESPFormulation):
    def __init__(self, instance: PESPInstance, env: gp.Env) -> None:
        super().__init__(env)

        self._T   = instance.T
        lb, ub, w = instance.lower_bound, instance.upper_bound, instance.weight

        # Events are 0-based, so from_event/to_event are already column indices for B.
        fi, ti  = instance.from_event, instance.to_event
        arc_idx = np.arange(instance.n_arcs)

        # Arc-node incidence matrix B (n_arcs × n_events): B[a,j]=+1, B[a,i]=-1
        B = csr_matrix(
            (np.r_[np.ones(instance.n_arcs), -np.ones(instance.n_arcs)],
             (np.r_[arc_idx, arc_idx], np.r_[ti, fi])),
            shape=(instance.n_arcs, instance.n_events),
        )

        # Tightest integer bounds on p_a given x ∈ [0, T)
        p_lb = np.ceil((lb - (self._T - 1)) / self._T).astype(int)
        p_ub = np.floor((ub + (self._T - 1)) / self._T).astype(int)

        # Decision variables
        z       = self._model.addMVar(instance.n_arcs, lb=lb, ub=ub, name="z")
        self._x = self._model.addMVar(instance.n_events, lb=0.0, ub=self._T - 1.0, name="x")
        p       = self._model.addMVar(instance.n_arcs, lb=p_lb, ub=p_ub, vtype=GRB.INTEGER, name="p")

        # Constraints
        self._model.addConstr(z == B @ self._x + self._T * p, name="link")
        self._model.setObjective(w @ z - float(w @ lb), GRB.MINIMIZE)

    def _extract(self) -> Timetable:
        return Timetable(np.round(self._x.X).astype(int), instance)
    
# Simple wrapper: solve a PESP instance via the arc-based formulation
def solve_pesp(instance: PESPInstance, env: gp.Env) -> Timetable:    
    with ArcFormulation(instance, env) as f:
        return f.solve()

### Testing
Let's see if this approach is able to solve our sample instance! Below we show the full lifecycle of reading an instance, solving the problem with our arc-based approach and validating the outcome. You should see an optimal objective value of 62, which is better than our initial solution above.

In [94]:
instance = parse_instance("data/small.txt", T=30)

with gp.Env() as env:
    timetable = solve_pesp(instance, env)

print(timetable)

obj, feas = timetable.evaluate()
print(f"objective={obj:.0f}, feasible={feas}")

Set parameter LicenseID to value 2664021
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 10 rows, 27 columns and 40 nonzeros (Min)
Model fingerprint: 0x35ad8b3d
Model has 10 linear objective coefficients and an objective constant of -323
Variable types: 17 continuous, 10 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+00, 4e+00]
  Bounds range     [1e+00, 3e+01]
  RHS range        [0e+00, 0e+00]

Presolve time: 0.00s
Presolved: 10 rows, 27 columns, 40 nonzeros
Variable types: 17 continuous, 10 integer (10 binary)
Found heuristic solution: objective 62.0000000

Root relaxation: objective 0.000000e+00, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   

## The cycle-based approach
### Motivation
The arc formulation above works correctly, but it has one integer variable $p_a$ for every arc. Since real-world railway timetables often require thousands (if not more) arcs, this formulation quickly becomes very difficult to solve. For that reason, another formulation was invented which focuses on **cycles**.

### Mathematical foundation
Let's look at any directed cycle in the constraint graph: a sequence of arcs that starts and ends at the same event. For each of these arcs, we have a constraint in the arc-based formulation: $x_j - x_i + T p_a \in [l_a, u_a]$. When you sum these constraints around the whole cycle, the event times cancel out and you are left with a purely integer constraint on the $p_a$ values. This is a **cycle constraint**: it says the $p_a$ values on the cycle cannot be chosen freely — they must be consistent.

Below is a specific example of a cycle. The constraint for the arc at the top would read $x_1 - x_0 + T p_0 \in [8,13]$. If we sum the four constraints involved in this cycle, we get $T \cdot (p_0 + p_4 + p_5 + p_6) \in [47, 68]$. Because all $p_a$ are integer, this can be simplified to $p_0 + p_4 + p_5 + p_6 = 2$. We call that the winding number for this particular cycle. If we "walk around" the cycle, we end up spending $2 \cdot T = 60$ minutes.

![Cycle winding](images/cycle_winding.png)

It has been shown that if we carefully select the right cycles in any given PESP graph and derive the constraints on the winding numbers for those cycles, the resulting MIP model is equivalent to the original formulation but often performs better. We will not prove exactly why this is the case or what the conditions for equivalence are. Instead we show a common way of selecting those cycles and formulating the resulting model.

We will start by constructing a spanning tree for the PESP graph. Such a tree has $|V|-1$ arcs by definition. Adding any of the remaining arcs to the tree would create a cycle. We call the set of those cycles the **fundamental cycles**. Those are exactly the cycles we need for our optimization model. And since there will be $|A| - |V| + |1|$ of those arcs, we often end up with a much smaller model that is easier to solve. For example, in our sample instance we have $10 - 7 + 1 = 4$ constraints instead of 10.

### Constructing the spanning tree

Finding a spanning tree is relatively simple. We first turn our constraint graph into an undirected graph, because the direction of arcs does not matter for *finding* the spanning tree. However, while we construct the tree, we not only capture the collection of arcs in the tree but construct various other data structures that we will need later: the "parent" for each node, the arc connecting it to the parent and the direction of traversing that arc.

![Spanning tree](images/spanning_tree.png)

We first define the `SpanningTree` class for describing a spanning tree on the constraint graph. On this class we also define a single function for turning a given set of tensions into a full solution. This can be done relatively simple by assuming the root node is assigned $t=0$. From there, all other times can be derived from the tensions in the spanning tree.

In [95]:
from collections import defaultdict, deque

@dataclass
class SpanningTree:    
    tree_parent: dict[EventId, EventId | None]
    tree_arc: dict[EventId, tuple[ArcIndex, Direction]]

    @property
    def n_events(self) -> int:
        return len(self.tree_parent)

    @cached_property
    def tree_arc_topdown(self) -> dict[EventId, list[tuple[EventId, ArcIndex, Direction]]]:
        # tree_arc is keyed by child; invert it to children keyed by parent for top-down BFS.
        children: dict[EventId, list[tuple[EventId, ArcIndex, Direction]]] = defaultdict(list)
        for v, (arc_idx, d) in self.tree_arc.items():
            parent = self.tree_parent[v]
            assert parent is not None  # tree_arc only contains non-root events
            children[parent].append((v, arc_idx, d))
        return children
    
    @cached_property
    def root_id(self) -> EventId:
        return next(e for e, p in self.tree_parent.items() if p is None)
    

    def recover_times(self, instance: PESPInstance, z_vals: np.ndarray) -> Timetable:
        children = self.tree_arc_topdown
        root = self.root_id
        
        # Root is fixed at time 0; array initialised to zeros covers this automatically.
        event_time = np.zeros(len(self.tree_parent))
        queue: deque[EventId] = deque([root])

        # Propagate tensions down the tree.
        while queue:
            u = queue.popleft()
            for v, arc_idx, d in children[u]:
                event_time[v] = (event_time[u] + d * z_vals[arc_idx]) % instance.T
                queue.append(v)

        return Timetable(np.round(event_time).astype(int) % instance.T, instance)

Now we define our algorithm for constructing the spanning tree. We first create an undirected version of our graph. Then we consider the first event the root and apply breadth-first search until we've reached all nodes.


In [96]:
def find_spanning_tree(
    from_list: list[EventId],
    to_list: list[EventId],
) -> SpanningTree:
    # Treat each directed arc as undirected by adding both orientations.
    adj: dict[EventId, list[tuple[EventId, ArcIndex, Direction]]] = defaultdict(list)
    for a, (u, v) in enumerate(zip(from_list, to_list)):
        adj[u].append((v, a, +1))
        if u != v:
            adj[v].append((u, a, -1))

    tree_parent: dict[EventId, EventId | None] = {}
    tree_arc: dict[EventId, tuple[ArcIndex, Direction]] = {}

    root = from_list[0]
    tree_parent[root] = None
    queue: deque[EventId] = deque([root])
    while queue:
        u = queue.popleft()
        for v, a, d in adj[u]:
            if v not in tree_parent:
                tree_parent[v] = u
                tree_arc[v] = (a, d)
                queue.append(v)

    return SpanningTree(tree_parent=tree_parent, tree_arc=tree_arc)

Let's see if we can find a spanning tree for our small instance:

In [97]:
from_list = instance.from_event.tolist()
to_list   = instance.to_event.tolist()
tree = find_spanning_tree(from_list, to_list)

root = next(e for e, p in tree.tree_parent.items() if p is None)
print(f"Root: {root}")
print("Tree edges (parent → child via arc):")
for v in sorted(tree.tree_arc):
    arc_id, dir = tree.tree_arc[v]
    u = tree.tree_parent[v]
    f, t = from_list[arc_id], to_list[arc_id]
    print(f"  {u} → {v}   a{arc_id}  ({f}→{t}, d={dir:+d})")

Root: 0
Tree edges (parent → child via arc):
  0 → 1   a0  (0→1, d=+1)
  1 → 2   a1  (1→2, d=+1)
  1 → 3   a2  (3→1, d=-1)
  0 → 4   a4  (4→0, d=-1)
  1 → 5   a5  (1→5, d=+1)
  4 → 6   a8  (4→6, d=+1)


### Fundamental cycles
After we have constructed the spanning tree with $|V|-1$ tree arcs, we are left with $C$ **co-tree arcs** outside the tree. Each co-tree arc $e = (u, v)$ closes exactly one **fundamental cycle**: traverse $e$ forward from $u$ to $v$, then follow the unique tree path back from $v$ to $u$. That unique path visits the **lowest common ancestor** of $u$ and $v$ in the spanning tree. 

Each fundamental cycle is defined by the arcs being traversed, as well of the direction of traversal (with respect to how the original arc was defined). We can therefore describe a **cycle basis** as a cycle-arc incidence matrix. Each row $c$ describes a cycle, and the coefficients in that row are $+1$ for arcs traversed in their original direction and $-1$ for arcs traversed backward by the cycle. We store this matrix in Compressed Sparse Row form. From this matrix we can compute bounds which will be explained and used a bit later.

![Fundamental cycle](images/fundamental_cycle2.png)

The image above shows one fundamental cycle for the co-tree arc between nodes 6 and 5. From node 5, we follow the blue path through node 1 to the root 0, then back over the green path through node 4 to node 6. If we make the direction of co-tree arc leading, then arcs 5, 0 and 4 are traversed in the opposite direction. This cycle would therefore be encoded as `[-1, 0, 0, 0, -1, -1, 0, 0, 1, 1]`.

We first define a function for finding a fundamental cycle for a given co-tree arc and spanning tree. As above, we first walk from the co-tree arc up to the root of the tree. Then we do the same from the other side of the co-tree arc, but stopping as soon as we encounter an ancestor that was also on the first path. That ancestor is the lowest common ancestor. While we do this, we track for each arc whether we traverse it in the spanning tree direction or the opposite one.

In [98]:
def find_fundamental_cycle(
    u: EventId,
    v: EventId,
    arc_idx: ArcIndex,
    tree: SpanningTree,
) -> dict[ArcIndex, Direction]:
    # Walk from u to the root to collect all ancestors, used to detect the LCA.
    ancestors_of_u: set[EventId] = set()
    node = u
    while node is not None:
        ancestors_of_u.add(node)
        node = tree.tree_parent[node]

    # The co-tree arc defines the cycle's orientation (traversed in its natural direction).
    # Walk from v to the LCA, reversing stored directions; then from u to the LCA, matching them.
    result = {arc_idx: +1}
    node = v
    while node not in ancestors_of_u:
        a_i, d = tree.tree_arc[node]
        result[a_i] = -d
        node = tree.tree_parent[node]
    lca = node
    node = u
    while node != lca:
        a_i, d = tree.tree_arc[node]
        result[a_i] = d
        node = tree.tree_parent[node]

    return result

Next, we define a datastructure for describing a **cycle basis**. This basis is essentially a collection of fundamental cycles, described by their **cycle-arc incidence matrix**. There is an additional calculation `window_number_bounds` here, which will become clear once we start looking at the mathematical formulation for this cycle-based approach.

In [99]:
@dataclass
class CycleBasis:
    instance: PESPInstance
    cycle_matrix: csr_matrix
    tree: SpanningTree

    @property
    def n_cycles(self) -> int:
        return self.cycle_matrix.shape[0]

    @property
    def n_arcs(self) -> int:
        return self.cycle_matrix.shape[1]

    @property
    def n_events(self) -> int:
        return self.tree.n_events

    def winding_number_bounds(
        self, lb: np.ndarray, ub: np.ndarray, T: int
    ) -> tuple[np.ndarray, np.ndarray]:
        D_pos = self.cycle_matrix.maximum(0)
        D_neg = (-self.cycle_matrix).maximum(0)
        q_lb = np.ceil((D_pos @ lb - D_neg @ ub) / T).astype(int)
        q_ub = np.floor((D_pos @ ub - D_neg @ lb) / T).astype(int)
        return q_lb, q_ub

    def recover_times(self, z_vals: np.ndarray) -> Timetable:
        return self.tree.recover_times(instance, z_vals)

    def __repr__(self) -> str:
        return (
            f"CycleBasis(n_cycles={self.n_cycles}, n_arcs={self.n_arcs}, "
            f"n_events={self.n_events})"
        )

We now have all pieces of logic required to find a cycle basis. For a given PESP instance, we first construct a spanning tree. Then for the co-tree arcs, we find the fundamental cycles and use them to construct the signed cycle-arc incidence matrix. We then have everything we need to describe the cycle basis.

In [100]:
def generate_cycle_basis(instance: PESPInstance) -> CycleBasis:
    from_list = instance.from_event.tolist()
    to_list   = instance.to_event.tolist()

    # Grow a spanning tree by BFS and record the parent and arc for each node.
    tree = find_spanning_tree(from_list, to_list)

    # Co-tree arcs are those not selected into the spanning tree.
    tree_arc_set = {a for a, _ in tree.tree_arc.values()}
    co_tree = [a for a in range(instance.n_arcs) if a not in tree_arc_set]

    # Compute each fundamental cycle and collect its signed arc directions as COO entries.
    rows, cols, data = [], [], []
    for idx, arc_id in enumerate(co_tree):
        u, v = from_list[arc_id], to_list[arc_id]
        for a_i, dir in find_fundamental_cycle(u, v, arc_id, tree).items():
            rows.append(idx)
            cols.append(a_i)
            data.append(dir)

    # Assemble the signed cycle-arc incidence matrix in CSR format.
    D = csr_matrix(
        (np.array(data, dtype=np.int8), (rows, cols)),
        shape=(len(co_tree), instance.n_arcs),
    )

    return CycleBasis(instance, D, tree)

We can test this for our small instance. As shown before, we expect four cycles. We will later see how these are used to construct constraints.

In [101]:
cb = generate_cycle_basis(instance)

print(cb)
print(f"Integer saving: {instance.n_arcs} arcs → {cb.n_cycles} cycles "
      f"({instance.n_arcs - cb.n_cycles} fewer integers)")
D = cb.cycle_matrix.toarray()
print(D)

CycleBasis(n_cycles=4, n_arcs=10, n_events=7)
Integer saving: 10 arcs → 4 cycles (6 fewer integers)
[[ 0  1  1  1  0  0  0  0  0  0]
 [ 1  0  0  0  1  1  1  0  0  0]
 [ 0  0  1  0  0  1  0  1  0  0]
 [-1  0  0  0 -1 -1  0  0  1  1]]


### Mathematics of cycles and windings
Earlier we defined **arc tensions** as $z_a = x_j - x_i + T \cdot p_a \;\in\; [l_a,\; u_a]$. We now also have our cycle-arc incidence matrix which we will call $D$ with $d_{c,a}$ the direction of traversing arc $a$ in cycle $c$. We can use this to arrive at a different relationship, describing that for each fundamental cycle $c$, the signed sum of tensions around the cycle must be a multiple of $T$:

$$\sum_{a \in c} d_{ca} \cdot z_a = T \cdot q_c, \quad q_c \in \mathbb{Z}$$

Here, $q_c$ is the **winding number** we mentioned earlier - an integer counting how many full periods the cycle wraps around. Interestingly, it has been proven that if we use the cycle basis described earlier for picking our cycles, then a solution to this cycle-based formulation can always be transformed into a solution to the arc-based formulation. 

This relationship is part of the following formulation:

$$\min_{z,\,q}\; \mathbf{w}^\top(z - \mathbf{l}) \quad \\ \text{s.t.} \quad \mathbf{l} \;\le\; z \;\le\; \mathbf{u} \\ \quad Dz = Tq \\ \quad q \in \mathbb{Z}^C$$

This has $|A|$ continuous variables and only $C = |A| - |V| + 1$ integer variables — a saving of $|V| - 1$ integers compared to the arc-based formulation. After solving, event times are not directly available because the $x$ variables were eliminated. However, we can apply the logic we implemented in the `SpanningTree` class above to transform tensions into actual event times.

#### Derivation of bounds
Above we introduced $q_c$ as the **winding number** of cycle $c$. We can derive tight bounds for these variables by analyzing the corresponding row $d_c$ of matrix $D$. Feel free to skip this section.
- We know that $q_c \cdot T = d_c \cdot z$ from the constraint above.
- We also know that each arc tension $z_a \in [l_a, u_a]$.
- Therefore, every *forward* arc in the cycle, therefore contributes a tension within that interval.
- Every *backward* arc contributes between $-u_a$ and $-l_a$. 
- Now split $d_c$ into a positive part and negative part by $d_c = d^+_c - d^-_c$.
- Then, we can state $d^+_c l - d^-_c u \leq d_c z \leq d^+_c u - d^-_c l$
- And if we divide everything by $T$, the middle part becomes $q_c$ and we get the bounds:

$$q_c \;\in\; \left[\left\lceil \frac{(D^+\,\mathbf{l} - D^-\,\mathbf{u})_c}{T} \right\rceil,\; \left\lfloor \frac{(D^+\,\mathbf{u} - D^-\,\mathbf{l})_c}{T} \right\rfloor\right]$$

This logic is implemented in the `winding_number_bounds` function above. For our sample instance all four winding numbers are uniquely determined by the bounds (each interval contains exactly one integer), so the integer relaxation is already tight.

In [105]:
class CycleFormulation(PESPFormulation):
    def __init__(
        self,
        instance: PESPInstance,
        env: gp.Env
    ) -> None:
        super().__init__(env)

        self._cb  = generate_cycle_basis(instance)
        self._T   = instance.T
        lb, ub, w = instance.lower_bound, instance.upper_bound, instance.weight

        q_lb, q_ub = self._cb.winding_number_bounds(lb, ub, self._T)

        self._z = self._model.addMVar(self._cb.n_arcs,   lb=lb, ub=ub, name="z")
        q       = self._model.addMVar(self._cb.n_cycles, lb=q_lb, ub=q_ub, vtype=GRB.INTEGER, name="q")

        self._model.addConstr(self._cb.cycle_matrix @ self._z == self._T * q, name="cycle")
        self._model.setObjective(w @ self._z - float(w @ lb), GRB.MINIMIZE)

    def _extract(self) -> Timetable:
        return self._cb.recover_times(self._z.X)
    
def solve_pesp_cycle(instance: PESPInstance, env: gp.Env) -> Timetable:
    with CycleFormulation(instance, env) as f:
        return f.solve()

### Testing

We can now test this approach on the same small dataset. We obtain a solution with the same objective value.

In [106]:
with gp.Env() as env:
    timetable = solve_pesp_cycle(instance, env)

print(timetable)

obj, feas = timetable.evaluate()
print(f"objective={obj:.0f}, feasible={feas}")

Set parameter LicenseID to value 2664021
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 4 rows, 14 columns and 19 nonzeros (Min)
Model fingerprint: 0xbd394643
Model has 10 linear objective coefficients and an objective constant of -323
Variable types: 10 continuous, 4 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+00, 4e+00]
  Bounds range     [1e+00, 3e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 4 rows and 14 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 1: 62 

Optimal solution found (tolerance 1.00e-04)
Best objective 6.200000000000e+01, best bound 6.200000000000e+01, gap 0.0000%
Timetable(times=arr

### Practical example

We mentioned earlier that the generic PESP structure is flexible enough to formulate real-world timetabling problems. Below we show an example for a network with three train services (blue, green, orange). The blue line serves five stations: events 1 and 6 represent visits to the same station from both directions. Solid horizontal arcs indicate driving time constraints. We also include arcs at the end of the service, where the trains change direction and some turnaround time is required. Finally, dotted arcs show transfer constraints between services. These guarantee that passengers that need to transfer from one service to another, have an acceptable waiting time. The graph shows 24 events and 28 arcs. As you will see, Gurobi very quickly computes the optimal timetable for this graph.

![med](images/medium.png)

In [117]:
instance = parse_instance('data/medium.txt', T = 30)
with gp.Env() as env:
    timetable = solve_pesp_cycle(instance, env)

print(timetable)

obj, feas = timetable.evaluate()
print(f"objective={obj:.0f}, feasible={feas}")

Set parameter LicenseID to value 2664021
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 5 rows, 33 columns and 43 nonzeros (Min)
Model fingerprint: 0x4b2a91cc
Model has 22 linear objective coefficients and an objective constant of -98
Variable types: 28 continuous, 5 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [0e+00, 0e+00]

Presolve removed 5 rows and 33 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 1: 11 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.100000000000e+01, best bound 1.100000000000e+01, gap 0.0000%
Timetable(times=arra